# MegaFS Image Similarity Evaluation

This notebook evaluates face swapping results using various image similarity metrics including LPIPS, PSNR, SSIM, and MSE.

## Evaluation Metrics

- **LPIPS**: Learned Perceptual Image Patch Similarity (lower is better)
- **PSNR**: Peak Signal-to-Noise Ratio (higher is better)
- **SSIM**: Structural Similarity Index (higher is better, range [0,1])
- **MSE**: Mean Squared Error (lower is better)

## Setup Instructions

1. **Update repository URL** in the first cell below
2. **Upload your dataset** to Google Drive:
   - Upload `celeba_mask_hq.zip` to `/content/drive/MyDrive/Datasets/`
3. **Upload weight files** to Google Drive:
   - Place all weight files in `/content/drive/MyDrive/Datasets/weights/`
4. **Run all cells** - evaluation will be performed automatically

## Features

- **Comprehensive Metrics**: Multiple perceptual and pixel-wise metrics
- **Batch Evaluation**: Process multiple image pairs efficiently
- **Statistical Analysis**: Mean, std, min, max, median across all results
- **Visualization**: Charts and graphs for result analysis
- **Export Results**: Save results to JSON for further analysis


In [ ]:
# Setup and clone repository
import os
import sys
import subprocess
import shutil

# IMPORTANT: Update this URL with your actual GitHub repository
repo_url = "https://github.com/n01r1r/MegaFS.git"  # ⚠️ CHANGE THIS URL IF NEEDED ⚠️
repo_dir = "/content/MegaFS"

print("=" * 60)
print("MegaFS Image Similarity Evaluation - Automatic Setup")
print("=" * 60)
print(f"Repository URL: {repo_url}")
print(f"Target directory: {repo_dir}")
print("=" * 60)

# Clone the repository if not already present
if not os.path.exists(repo_dir):
    print("INFO: Cloning MegaFS repository...")
    try:
        subprocess.run(["git", "clone", repo_url, repo_dir], check=True)
        print("SUCCESS: Repository cloned successfully")
    except subprocess.CalledProcessError as e:
        print(f"ERROR: Failed to clone repository: {e}")
        print("Please check the repository URL and try again")
        print("Make sure the repository is public or you have access")
        sys.exit(1)
else:
    print("INFO: Repository already exists, updating...")
    try:
        subprocess.run(["git", "-C", repo_dir, "pull"], check=True)
        print("SUCCESS: Repository updated")
    except subprocess.CalledProcessError as e:
        print(f"WARNING: Failed to update repository: {e}")

# Add the cloned repository to Python path
sys.path.insert(0, repo_dir)

# Change to the repository directory
os.chdir(repo_dir)

print("SUCCESS: Repository setup complete")
print(f"INFO: Working directory: {os.getcwd()}")
print("=" * 60)


In [ ]:
# Import basic libraries
import zipfile
from glob import glob
from tqdm.notebook import tqdm
import torch
import cv2
import numpy as np
import argparse
import json
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from google.colab import drive
from IPython.display import display, Image
from google.colab import files

print("SUCCESS: Basic libraries imported")


In [ ]:
# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("SUCCESS: Google Drive mounted")
except Exception as e:
    print(f"ERROR: Google Drive mount failed: {e}")

# Dataset preparation
print("INFO: Preparing dataset...")
dataset_zip_path = "/content/drive/MyDrive/Datasets/celeba_mask_hq.zip"
base_dir = "/content/"


In [ ]:
# Install required packages
print("INFO: Installing required packages...")
import subprocess
import sys

# Install lpips and other required packages
packages_to_install = [
    "lpips>=0.1.4",
    "scikit-image>=0.18.0", 
    "matplotlib>=3.3.0",
    "seaborn>=0.11.0",
    "pandas>=1.3.0"
]

for package in packages_to_install:
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"SUCCESS: {package} installed")
    except subprocess.CalledProcessError as e:
        print(f"WARNING: Failed to install {package}: {e}")

# Import modularized MegaFS components
from config import Config, DEFAULT_CONFIGS
from models.megafs import MegaFS
from models.weight_loaders import verify_all_weights
from utils.debug_utils import check_system_requirements
from utils.data_utils import DataMapManager
from utils.metrics import ImageMetrics, FaceSwapEvaluator, save_evaluation_results, load_evaluation_results

print("SUCCESS: All imports complete - MegaFS Evaluation ready")


In [ ]:
# Extract dataset if zip file exists
if os.path.exists(dataset_zip_path):
    print(f"INFO: Extracting dataset from '{dataset_zip_path}'")
    with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_dir)
    print("SUCCESS: Dataset extraction complete")
else:
    print(f"WARNING: Dataset zip file not found at '{dataset_zip_path}'")
    print("Please ensure the dataset is uploaded to Google Drive")


In [ ]:
# Dataset configuration
dataset_root = os.path.join(base_dir, "CelebAMask-HQ")
img_dir = os.path.join(dataset_root, "CelebA-HQ-img")
mask_base_dir = os.path.join(dataset_root, "CelebAMask-HQ-mask-anno")
data_map_path = os.path.join(repo_dir, "data_map.json")

print(f"INFO: Dataset root: {dataset_root}")
print(f"INFO: Data map path: {data_map_path}")

# Initialize data manager
data_manager = DataMapManager(data_map_path)
data_map = data_manager.data_map
valid_ids = []


In [ ]:
# Check data map status and get valid IDs
print("INFO: Checking data map status...")
if not os.path.exists(data_map_path):
    print(f"ERROR: Data map file '{data_map_path}' not found.")
    print("INFO: Generating data_map.json from dataset...")
    
    # Generate data map if it doesn't exist
    if os.path.exists(dataset_root):
        try:
            # Run create_datamap.py in the dataset root
            import subprocess
            result = subprocess.run([
                "python", "create_datamap.py"
            ], cwd=dataset_root, capture_output=True, text=True)
            
            if result.returncode == 0:
                print("SUCCESS: data_map.json generated")
                # Reload data manager
                data_manager = DataMapManager(data_map_path)
                data_map = data_manager.data_map
            else:
                print(f"ERROR: Failed to generate data_map.json: {result.stderr}")
        except Exception as e:
            print(f"ERROR: Failed to generate data_map.json: {e}")
    else:
        print("ERROR: Dataset root not found. Please check dataset extraction.")
else:
    print("SUCCESS: Found data_map.json")
    print(f"INFO: Loaded {len(data_map)} entries from data map")

# Get valid IDs for evaluation
print("INFO: Getting valid dataset IDs...")
valid_ids = data_manager.get_valid_ids(dataset_root, sample_size=1000)  # Larger sample for evaluation
print(f"SUCCESS: Found {len(valid_ids)} valid IDs")


In [ ]:
# System requirements check
print("INFO: Checking system requirements...")
check_system_requirements()

# Configuration setup
print("INFO: Setting up configuration...")

# Use evaluation configuration for Google Colab
config = DEFAULT_CONFIGS["evaluation"]

print("SUCCESS: Configuration created")
config.print_config()


In [ ]:
# Verify weight files
print("INFO: Verifying weight files...")
if not verify_all_weights(config.paths.checkpoint_dir):
    print("ERROR: Weight verification failed. Please check your weight files.")
    print("Required files:")
    print("  - ftm_final.pth")
    print("  - injection_final.pth") 
    print("  - lcr_final.pth")
    print("  - stylegan2-ffhq-config-f.pth")
else:
    print("SUCCESS: All weight files verified")

# Initialize MegaFS
print("INFO: Initializing MegaFS...")
handler = None

try:
    # Initialize MegaFS with configuration and data map
    handler = MegaFS(
        config=config,
        data_map=data_map,
        debug=True  # Enable debug logging
    )
    print(f"SUCCESS: {config.swap.swap_type}-MegaFS model handler created")
    
except Exception as e:
    print(f"ERROR: Failed to initialize MegaFS: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Initialize evaluation components
print("INFO: Initializing evaluation components...")

# Initialize metrics calculator
metrics_calculator = ImageMetrics(use_gpu=True)
print("SUCCESS: Image metrics calculator initialized")

# Initialize face swap evaluator
evaluator = FaceSwapEvaluator(use_gpu=True)
print("SUCCESS: Face swap evaluator initialized")

# Create results directory
results_dir = "/content/evaluation_results"
os.makedirs(results_dir, exist_ok=True)
print(f"SUCCESS: Results directory created: {results_dir}")


In [ ]:
def run_evaluation_pair(handler_instance, src_id, tgt_id, refine=True):
    """Run face swap and evaluation for a single image pair"""
    if not handler_instance:
        print("ERROR: Handler not initialized")
        return None

    print(f"INFO: Processing pair - Source ID: {src_id}, Target ID: {tgt_id}")
    
    try:
        # Run face swap
        result_path, result_image = handler_instance.run(
            src_idx=src_id,
            tgt_idx=tgt_id,
            refine=refine,
            save_path=f"{results_dir}/swap_result_{src_id}_to_{tgt_id}.jpg"
        )
        
        if result_image is None:
            print(f"ERROR: Failed to generate result for pair ({src_id}, {tgt_id})")
            return None
        
        # Extract individual images from result
        # Result format: [source, target, swapped, refined] horizontally concatenated
        img_width = result_image.shape[1] // 4 if refine else result_image.shape[1] // 3
        
        source_img = result_image[:, :img_width]
        target_img = result_image[:, img_width:img_width*2]
        swapped_img = result_image[:, img_width*2:img_width*3]
        refined_img = result_image[:, img_width*3:] if refine else None
        
        # Evaluate the results
        evaluation_results = evaluator.evaluate_pair(
            source_img, target_img, swapped_img, refined_img
        )
        
        # Add metadata
        evaluation_results['metadata'] = {
            'source_id': src_id,
            'target_id': tgt_id,
            'refine': refine,
            'result_path': result_path
        }
        
        print(f"SUCCESS: Evaluation completed for pair ({src_id}, {tgt_id})")
        return evaluation_results
        
    except Exception as e:
        print(f"ERROR: Evaluation failed for pair ({src_id}, {tgt_id}): {e}")
        return None


def run_batch_evaluation(handler_instance, id_pairs, refine=True, max_pairs=None):
    """Run evaluation for multiple image pairs"""
    if not handler_instance:
        print("ERROR: Handler not initialized")
        return []
    
    if max_pairs:
        id_pairs = id_pairs[:max_pairs]
    
    print(f"INFO: Starting batch evaluation for {len(id_pairs)} pairs...")
    
    all_results = []
    failed_pairs = []
    
    for i, (src_id, tgt_id) in enumerate(tqdm(id_pairs, desc="Batch evaluation")):
        try:
            result = run_evaluation_pair(handler_instance, src_id, tgt_id, refine)
            if result:
                all_results.append(result)
            else:
                failed_pairs.append((src_id, tgt_id))
                
        except Exception as e:
            print(f"ERROR: Failed to process pair ({src_id}, {tgt_id}): {e}")
            failed_pairs.append((src_id, tgt_id))
            continue
    
    print(f"SUCCESS: Batch evaluation completed")
    print(f"  - Successful pairs: {len(all_results)}")
    print(f"  - Failed pairs: {len(failed_pairs)}")
    
    if failed_pairs:
        print(f"  - Failed pair IDs: {failed_pairs[:10]}...")  # Show first 10 failed pairs
    
    return all_results

print("SUCCESS: Evaluation functions defined")


## Evaluation Execution

The following cells run the actual evaluation experiments. You can modify the parameters as needed:

- **Evaluation Size**: Number of image pairs to evaluate
- **Swap Types**: Different face swapping methods to compare
- **Refinement**: Whether to include refinement step


In [ ]:
# Evaluation configuration
EVALUATION_SIZE = 50  # Number of pairs to evaluate (adjust as needed)
REFINE_ENABLED = True  # Whether to include refinement step

print(f"INFO: Evaluation configuration:")
print(f"  - Evaluation size: {EVALUATION_SIZE} pairs")
print(f"  - Refinement enabled: {REFINE_ENABLED}")
print(f"  - Available valid IDs: {len(valid_ids)}")

# Generate random pairs for evaluation
import random
random.seed(42)  # For reproducible results

if len(valid_ids) >= 2:
    # Create random pairs from valid IDs
    evaluation_pairs = []
    for _ in range(EVALUATION_SIZE):
        src_id = random.choice(valid_ids)
        tgt_id = random.choice(valid_ids)
        while tgt_id == src_id:  # Ensure different IDs
            tgt_id = random.choice(valid_ids)
        evaluation_pairs.append((src_id, tgt_id))
    
    print(f"SUCCESS: Generated {len(evaluation_pairs)} evaluation pairs")
    print(f"Sample pairs: {evaluation_pairs[:5]}")
else:
    print("ERROR: Not enough valid IDs for evaluation")
    evaluation_pairs = []


In [ ]:
# Run evaluation for FTM method
print("INFO: Running evaluation for FTM method...")

if handler and evaluation_pairs:
    # Run batch evaluation
    ftm_results = run_batch_evaluation(
        handler_instance=handler,
        id_pairs=evaluation_pairs,
        refine=REFINE_ENABLED,
        max_pairs=EVALUATION_SIZE
    )
    
    if ftm_results:
        print(f"SUCCESS: FTM evaluation completed with {len(ftm_results)} results")
        
        # Save results
        ftm_results_path = os.path.join(results_dir, "ftm_evaluation_results.json")
        save_evaluation_results(ftm_results, ftm_results_path)
        
        # Calculate statistics
        ftm_stats = evaluator.calculate_statistics(ftm_results)
        ftm_stats_path = os.path.join(results_dir, "ftm_evaluation_stats.json")
        save_evaluation_results(ftm_stats, ftm_stats_path)
        
        print(f"SUCCESS: FTM results saved to {ftm_results_path}")
        print(f"SUCCESS: FTM statistics saved to {ftm_stats_path}")
    else:
        print("ERROR: FTM evaluation failed - no results generated")
        ftm_results = []
else:
    print("ERROR: Cannot run FTM evaluation - handler not initialized or no evaluation pairs")
    ftm_results = []


In [ ]:
# Run evaluation for Injection method
print("INFO: Running evaluation for Injection method...")

if evaluation_pairs:
    try:
        # Create new config for injection method
        injection_config = Config(
            swap_type="injection",
            dataset_root=config.paths.dataset_root,
            img_root=config.paths.img_root,
            mask_root=config.paths.mask_root,
            checkpoint_dir=config.paths.checkpoint_dir
        )
        
        # Initialize injection handler
        injection_handler = MegaFS(
            config=injection_config,
            data_map=data_map,
            debug=False  # Reduce debug output for batch processing
        )
        
        print("SUCCESS: Injection handler initialized")
        
        # Run batch evaluation
        injection_results = run_batch_evaluation(
            handler_instance=injection_handler,
            id_pairs=evaluation_pairs,
            refine=REFINE_ENABLED,
            max_pairs=EVALUATION_SIZE
        )
        
        if injection_results:
            print(f"SUCCESS: Injection evaluation completed with {len(injection_results)} results")
            
            # Save results
            injection_results_path = os.path.join(results_dir, "injection_evaluation_results.json")
            save_evaluation_results(injection_results, injection_results_path)
            
            # Calculate statistics
            injection_stats = evaluator.calculate_statistics(injection_results)
            injection_stats_path = os.path.join(results_dir, "injection_evaluation_stats.json")
            save_evaluation_results(injection_stats, injection_stats_path)
            
            print(f"SUCCESS: Injection results saved to {injection_results_path}")
            print(f"SUCCESS: Injection statistics saved to {injection_stats_path}")
        else:
            print("ERROR: Injection evaluation failed - no results generated")
            injection_results = []
            
    except Exception as e:
        print(f"ERROR: Failed to run injection evaluation: {e}")
        injection_results = []
else:
    print("ERROR: Cannot run injection evaluation - no evaluation pairs")
    injection_results = []


In [ ]:
# Run evaluation for LCR method
print("INFO: Running evaluation for LCR method...")

if evaluation_pairs:
    try:
        # Create new config for LCR method
        lcr_config = Config(
            swap_type="lcr",
            dataset_root=config.paths.dataset_root,
            img_root=config.paths.img_root,
            mask_root=config.paths.mask_root,
            checkpoint_dir=config.paths.checkpoint_dir
        )
        
        # Initialize LCR handler
        lcr_handler = MegaFS(
            config=lcr_config,
            data_map=data_map,
            debug=False  # Reduce debug output for batch processing
        )
        
        print("SUCCESS: LCR handler initialized")
        
        # Run batch evaluation
        lcr_results = run_batch_evaluation(
            handler_instance=lcr_handler,
            id_pairs=evaluation_pairs,
            refine=REFINE_ENABLED,
            max_pairs=EVALUATION_SIZE
        )
        
        if lcr_results:
            print(f"SUCCESS: LCR evaluation completed with {len(lcr_results)} results")
            
            # Save results
            lcr_results_path = os.path.join(results_dir, "lcr_evaluation_results.json")
            save_evaluation_results(lcr_results, lcr_results_path)
            
            # Calculate statistics
            lcr_stats = evaluator.calculate_statistics(lcr_results)
            lcr_stats_path = os.path.join(results_dir, "lcr_evaluation_stats.json")
            save_evaluation_results(lcr_stats, lcr_stats_path)
            
            print(f"SUCCESS: LCR results saved to {lcr_results_path}")
            print(f"SUCCESS: LCR statistics saved to {lcr_stats_path}")
        else:
            print("ERROR: LCR evaluation failed - no results generated")
            lcr_results = []
            
    except Exception as e:
        print(f"ERROR: Failed to run LCR evaluation: {e}")
        lcr_results = []
else:
    print("ERROR: Cannot run LCR evaluation - no evaluation pairs")
    lcr_results = []


## Results Analysis and Visualization

The following cells analyze the evaluation results and create visualizations to compare different face swapping methods.


In [ ]:
# Load and analyze results
print("INFO: Loading evaluation results...")

all_results = {}
all_stats = {}

# Load FTM results
if 'ftm_results' in locals() and ftm_results:
    all_results['FTM'] = ftm_results
    if 'ftm_stats' in locals():
        all_stats['FTM'] = ftm_stats
    print(f"SUCCESS: FTM results loaded ({len(ftm_results)} pairs)")

# Load Injection results
if 'injection_results' in locals() and injection_results:
    all_results['Injection'] = injection_results
    if 'injection_stats' in locals():
        all_stats['Injection'] = injection_stats
    print(f"SUCCESS: Injection results loaded ({len(injection_results)} pairs)")

# Load LCR results
if 'lcr_results' in locals() and lcr_results:
    all_results['LCR'] = lcr_results
    if 'lcr_stats' in locals():
        all_stats['LCR'] = lcr_stats
    print(f"SUCCESS: LCR results loaded ({len(lcr_results)} pairs)")

print(f"INFO: Total methods evaluated: {len(all_results)}")
print(f"Methods: {list(all_results.keys())}")

# Create comparison summary
if all_stats:
    print("\n" + "="*60)
    print("EVALUATION SUMMARY")
    print("="*60)
    
    for method, stats in all_stats.items():
        print(f"\n{method} Method:")
        for comparison_type, metrics in stats.items():
            if comparison_type != 'metadata':
                print(f"  {comparison_type}:")
                for metric_name, stat_values in metrics.items():
                    if stat_values['count'] > 0:
                        print(f"    {metric_name}: {stat_values['mean']:.4f} ± {stat_values['std']:.4f}")
                    else:
                        print(f"    {metric_name}: No data")
else:
    print("WARNING: No statistics available for comparison")


In [ ]:
# Create visualizations
print("INFO: Creating visualizations...")

if all_stats and len(all_stats) > 1:
    # Set up the plotting style
    plt.style.use('default')
    sns.set_palette("husl")
    
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('Face Swapping Methods Comparison', fontsize=16, fontweight='bold')
    
    # Define metrics to plot
    metrics_to_plot = ['lpips', 'psnr', 'ssim', 'mse']
    metric_titles = ['LPIPS (Lower Better)', 'PSNR (Higher Better)', 'SSIM (Higher Better)', 'MSE (Lower Better)']
    comparison_types = ['swapped_vs_target', 'refined_vs_target']
    
    for idx, (metric, title) in enumerate(zip(metrics_to_plot, metric_titles)):
        row = idx // 2
        col = idx % 2
        ax = axes[row, col]
        
        # Prepare data for plotting
        plot_data = []
        methods = []
        
        for method, stats in all_stats.items():
            for comp_type in comparison_types:
                if comp_type in stats and metric in stats[comp_type]:
                    value = stats[comp_type][metric]['mean']
                    if not np.isnan(value) and value != float('inf'):
                        plot_data.append(value)
                        methods.append(f"{method}\n({comp_type.replace('_', ' ').title()})")
        
        if plot_data:
            # Create bar plot
            bars = ax.bar(range(len(plot_data)), plot_data, alpha=0.7)
            ax.set_title(title, fontweight='bold')
            ax.set_ylabel('Value')
            ax.set_xticks(range(len(plot_data)))
            ax.set_xticklabels(methods, rotation=45, ha='right')
            
            # Add value labels on bars
            for bar, value in zip(bars, plot_data):
                height = bar.get_height()
                ax.text(bar.get_x() + bar.get_width()/2., height + height*0.01,
                       f'{value:.3f}', ha='center', va='bottom', fontsize=8)
            
            # Add grid for better readability
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No data available', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(title, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    print("SUCCESS: Comparison plots created")
else:
    print("WARNING: Not enough data for comparison plots")


In [ ]:
# Create detailed statistics table
print("INFO: Creating detailed statistics table...")

if all_stats:
    # Create a comprehensive statistics table
    stats_data = []
    
    for method, stats in all_stats.items():
        for comparison_type, metrics in stats.items():
            if comparison_type != 'metadata':
                for metric_name, stat_values in metrics.items():
                    if stat_values['count'] > 0:
                        stats_data.append({
                            'Method': method,
                            'Comparison': comparison_type.replace('_', ' ').title(),
                            'Metric': metric_name.upper(),
                            'Mean': stat_values['mean'],
                            'Std': stat_values['std'],
                            'Min': stat_values['min'],
                            'Max': stat_values['max'],
                            'Median': stat_values['median'],
                            'Count': stat_values['count']
                        })
    
    if stats_data:
        # Create DataFrame
        df_stats = pd.DataFrame(stats_data)
        
        # Display the table
        print("\n" + "="*100)
        print("DETAILED STATISTICS TABLE")
        print("="*100)
        
        # Format the DataFrame for better display
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', None)
        pd.set_option('display.max_colwidth', None)
        
        print(df_stats.to_string(index=False, float_format='%.4f'))
        
        # Save to CSV
        csv_path = os.path.join(results_dir, "detailed_statistics.csv")
        df_stats.to_csv(csv_path, index=False)
        print(f"\nSUCCESS: Detailed statistics saved to {csv_path}")
        
        # Create summary by method
        print("\n" + "="*80)
        print("SUMMARY BY METHOD")
        print("="*80)
        
        for method in df_stats['Method'].unique():
            method_data = df_stats[df_stats['Method'] == method]
            print(f"\n{method} Method:")
            
            # Group by comparison type
            for comp_type in method_data['Comparison'].unique():
                comp_data = method_data[method_data['Comparison'] == comp_type]
                print(f"  {comp_type}:")
                
                for _, row in comp_data.iterrows():
                    print(f"    {row['Metric']}: {row['Mean']:.4f} ± {row['Std']:.4f} (n={row['Count']})")
    else:
        print("WARNING: No statistics data available for table creation")
else:
    print("WARNING: No statistics available for table creation")


In [ ]:
# Results file management and download
print("INFO: Managing result files...")

# List all result files
result_files = []
for root, dirs, files in os.walk(results_dir):
    for file in files:
        if file.endswith(('.json', '.csv', '.jpg')):
            result_files.append(os.path.join(root, file))

if result_files:
    print(f"SUCCESS: Found {len(result_files)} result files:")
    for file in result_files:
        file_size = os.path.getsize(file) / (1024 * 1024)  # Size in MB
        print(f"  - {os.path.basename(file)} ({file_size:.2f} MB)")
    
    # Create a summary report
    summary_report = f"""
# MegaFS Face Swapping Evaluation Report

## Evaluation Configuration
- Evaluation Size: {EVALUATION_SIZE} pairs
- Refinement Enabled: {REFINE_ENABLED}
- Methods Evaluated: {', '.join(all_results.keys()) if all_results else 'None'}

## Metrics Evaluated
- LPIPS: Learned Perceptual Image Patch Similarity (lower is better)
- PSNR: Peak Signal-to-Noise Ratio (higher is better)
- SSIM: Structural Similarity Index (higher is better, range [0,1])
- MSE: Mean Squared Error (lower is better)

## Results Summary
"""
    
    if all_stats:
        for method, stats in all_stats.items():
            summary_report += f"\n### {method} Method\n"
            for comparison_type, metrics in stats.items():
                if comparison_type != 'metadata':
                    summary_report += f"\n#### {comparison_type.replace('_', ' ').title()}\n"
                    for metric_name, stat_values in metrics.items():
                        if stat_values['count'] > 0:
                            summary_report += f"- {metric_name.upper()}: {stat_values['mean']:.4f} ± {stat_values['std']:.4f}\n"
    
    summary_report += f"""
## Files Generated
{chr(10).join([f"- {os.path.basename(f)}" for f in result_files])}

## Download Instructions
To download all result files, run the following commands in separate cells:
"""
    
    for file in result_files:
        summary_report += f"files.download('{file}')\n"
    
    # Save summary report
    summary_path = os.path.join(results_dir, "evaluation_report.md")
    with open(summary_path, 'w') as f:
        f.write(summary_report)
    
    print(f"SUCCESS: Summary report saved to {summary_path}")
    
    # Display summary
    print("\n" + "="*80)
    print("EVALUATION COMPLETE")
    print("="*80)
    print(f"Total files generated: {len(result_files)}")
    print(f"Results directory: {results_dir}")
    print(f"Summary report: {summary_path}")
    
    print("\nTo download files, run the following commands:")
    for file in result_files[:5]:  # Show first 5 files
        print(f"files.download('{file}')")
    if len(result_files) > 5:
        print(f"... and {len(result_files) - 5} more files")
        
else:
    print("WARNING: No result files found")


## Additional Analysis (Optional)

The following cells provide additional analysis capabilities that you can run if needed.


In [ ]:
# Optional: Single pair detailed analysis
def analyze_single_pair(src_id, tgt_id, method="ftm"):
    print(f"INFO: Analyzing single pair - Source: {src_id}, Target: {tgt_id}, Method: {method}")
    
    # Create config for the specified method
    single_config = Config(
        swap_type=method,
        dataset_root=config.paths.dataset_root,
        img_root=config.paths.img_root,
        mask_root=config.paths.mask_root,
        checkpoint_dir=config.paths.checkpoint_dir
    )
    
    try:
        # Initialize handler
        single_handler = MegaFS(
            config=single_config,
            data_map=data_map,
            debug=False
        )
        
        # Run evaluation
        result = run_evaluation_pair(single_handler, src_id, tgt_id, refine=True)
        
        if result:
            print("SUCCESS: Single pair analysis completed")
            
            # Display detailed results
            print("\nDetailed Results:")
            for comparison_type, metrics in result.items():
                if comparison_type != 'metadata':
                    print(f"\n{comparison_type.replace('_', ' ').title()}:")
                    for metric_name, value in metrics.items():
                        print(f"  {metric_name.upper()}: {value:.6f}")
            
            return result
        else:
            print("ERROR: Single pair analysis failed")
            return None
            
    except Exception as e:
        print(f"ERROR: Single pair analysis failed: {e}")
        return None

# Example usage (uncomment to run):
# single_result = analyze_single_pair(100, 200, "ftm")


In [ ]:
# Optional: Custom evaluation with different parameters
def run_custom_evaluation(swap_type="ftm", num_pairs=20, refine=True):
    print(f"INFO: Running custom evaluation - Method: {swap_type}, Pairs: {num_pairs}, Refine: {refine}")
    
    # Generate custom pairs
    import random
    random.seed(42)
    
    if len(valid_ids) >= 2:
        custom_pairs = []
        for _ in range(num_pairs):
            src_id = random.choice(valid_ids)
            tgt_id = random.choice(valid_ids)
            while tgt_id == src_id:
                tgt_id = random.choice(valid_ids)
            custom_pairs.append((src_id, tgt_id))
        
        # Create config
        custom_config = Config(
            swap_type=swap_type,
            dataset_root=config.paths.dataset_root,
            img_root=config.paths.img_root,
            mask_root=config.paths.mask_root,
            checkpoint_dir=config.paths.checkpoint_dir
        )
        
        try:
            # Initialize handler
            custom_handler = MegaFS(
                config=custom_config,
                data_map=data_map,
                debug=False
            )
            
            # Run evaluation
            custom_results = run_batch_evaluation(
                handler_instance=custom_handler,
                id_pairs=custom_pairs,
                refine=refine,
                max_pairs=num_pairs
            )
            
            if custom_results:
                print(f"SUCCESS: Custom evaluation completed with {len(custom_results)} results")
                
                # Calculate and display statistics
                custom_stats = evaluator.calculate_statistics(custom_results)
                
                print(f"\nCustom Evaluation Results for {swap_type.upper()}:")
                for comparison_type, metrics in custom_stats.items():
                    if comparison_type != 'metadata':
                        print(f"\n{comparison_type.replace('_', ' ').title()}:")
                        for metric_name, stat_values in metrics.items():
                            if stat_values['count'] > 0:
                                print(f"  {metric_name.upper()}: {stat_values['mean']:.4f} ± {stat_values['std']:.4f}")
                
                return custom_results, custom_stats
            else:
                print("ERROR: Custom evaluation failed - no results generated")
                return [], {}
                
        except Exception as e:
            print(f"ERROR: Custom evaluation failed: {e}")
            return [], {}
    else:
        print("ERROR: Not enough valid IDs for custom evaluation")
        return [], {}

# Example usage (uncomment to run):
# custom_results, custom_stats = run_custom_evaluation("ftm", 10, True)
